# **Checkpointing in CrewAI**

## Automatically save execution state so crews, flows, and agents can resume after failures.

## Checkpointing saves a snapshot of execution state during a run so a crew, flow, or agent can resume after a failure or be forked into an alternate branch.


## What a checkpoint is
A checkpoint captures everything CrewAI needs to recreate a run mid-flight: the full state of the crew, flow, or agent — configuration, agent memory and knowledge sources, task progress, intermediate outputs, internal state and attributes — alongside the kickoff inputs, the event history up to that point, and a lineage ID that ties the checkpoint to the run it came from.
Restoring rebuilds that state and continues. Completed tasks are skipped, memory and knowledge are rehydrated, and downstream work runs against the same outputs the original run produced. Forking does the same restore under a new lineage, so the new branch and the original run can write checkpoints side by side without overwriting each other.

## When checkpoints are written
Checkpointing is **event-driven**.
The runtime subscribes to events you select via on_events and writes a checkpoint each time one fires.

The default task_completed produces one checkpoint per finished task — a sensible tradeoff between granularity and disk use.

Higher-frequency events like llm_call_completed are available for fine-grained recovery but write far more files.

## Storage
Two providers ship with CrewAI:

**JsonProvider** writes one file per checkpoint. Human-readable and easy to inspect.

**SqliteProvider** writes to a single SQLite database. Better for high-frequency checkpointing.
Both prune oldest checkpoints when max_checkpoints is set.

# Install Libraries

In [8]:
!pip install -q crewai

# Set API Keys for LLMs & Tools

In [4]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY_NEW')

# Import necessary classes/methods from the package

In [2]:
from crewai import Agent, Task, Crew, Process, LLM

# Define the LLM Object

In [5]:
# Create an LLM with a temperature of 0 to ensure deterministic outputs from the LLM
# Set temperature to higher values for creative outputs from the LLM

# OPENAI LLMs
llm = LLM(
          model="gpt-5.4-nano",
          base_url="https://api.openai.com/v1",
          api_key = os.environ["OPENAI_API_KEY"],
          temperature=0.7)


# # GROQ hosted LLMs
# llm = LLM(
#     #  model="qwen/qwen3.6-27b",
#      model="openai/gpt-oss-120b",
#      base_url="https://api.groq.com/openai/v1",
#      api_key=os.environ["GROQ_API_KEY"],
#      temperature=0.7)


# Define Agents, Tasks and Crew

In [6]:
from crewai import Agent, Crew, Task
import asyncio, time


#-------------------------------------------------------------
# Define Agents
researcher = Agent(
    role="Researcher",
    goal="Research",
    backstory="Expert",
    llm=llm)

writer = Agent(
    role="Writer",
    goal="Write",
    backstory="Expert",
    llm=llm)

#-------------------------------------------------------------
# Define a callback function for the Crew Tasks
# This callback simulates a delay for 10 seconds
# This callback runs after completion of each Task in the Crew

def crew_task_callback(output):
    print("Crew Callback :: Pausing for 10 seconds...")
    time.sleep(10)
    return output

#-------------------------------------------------------------
# Define Tasks
research_task = Task(
    description="Research AI trends",
    agent=researcher,
    expected_output="bullets",
    asynchronous=False
)

write_task = Task(
    description="Write a summary",
    agent=writer,
    expected_output="paragraph",
    context=[research_task]
)

#-------------------------------------------------------------
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    task_callback = crew_task_callback, #<= This callback ensures a pause for 10 seconds after the completion of a task
    checkpoint=True, #<= This enables checkpointing
)

Execute the following cell to run the crew.

Press **Ctrl+C** or **STOP** after the first task finishes. Check the message from the Crew's Task callback, as soon as the first message is printed, stop the execution of the next coding cell.

Look in **./.checkpoints/** — a file named <timestamp>_<uuid>.json is the checkpoint.

In [7]:
result = await crew.kickoff_async()
result

Crew Callback :: Pausing for 10 seconds...


CancelledError: 

In [9]:
# As we terminated the execution at the middle, result has not been instantiated yet
result

NameError: name 'result' is not defined

## Discover what's in the checkpoint directory

In [3]:
# If you need to delete the .checkpoints/ directory for some reason
# !rm -rf .checkpoints/

In [12]:
!ls -al .checkpoints/

total 12
drwxr-xr-x 3 root root 4096 Aug 18 14:06 .
drwxr-xr-x 1 root root 4096 Aug 18 14:06 ..
drwxr-xr-x 2 root root 4096 Aug 18 14:07 main


In [13]:
!ls .checkpoints/main/*.json

.checkpoints/main/20260818T140652_3a8f7029_p-none.json
.checkpoints/main/20260818T140706_4a43fbb4_p-20260818T140652_3a8f7029.json


Load the checkpint from the stored json file and run the crew again.

Update **restore_from="./.checkpoints/<timestamp>_<uuid>.json"**

With the name of the appropriate file.

In [14]:
os.environ["CREWAI_DESERIALIZE_CALLBACKS"] = "1"

In [15]:
from crewai import CheckpointConfig

result = await crew.kickoff_async(
    from_checkpoint=CheckpointConfig(
        # restore_from="./.checkpoints/<timestamp>_<uuid>.json",
        restore_from="./.checkpoints/main/20260818T140706_4a43fbb4_p-20260818T140652_3a8f7029.json"

    ),
)

print(result)

Across the industry, generative AI is rapidly evolving from simple text chat into practical, production-ready systems that combine multimodal understanding, agentic workflows, and stronger grounding. Multimodal models increasingly handle text, images, audio, and video together—enabling “AI copilots” that can interpret what users see or hear—not just what they type. In parallel, systems are shifting from single-shot responses to agentic architectures where an LLM plans, calls tools (search, code execution, databases, ticketing systems), tracks state, and iterates with reliability guardrails. Retrieval-Augmented Generation (RAG) remains central but is improving through better chunking, hybrid retrieval with reranking, structured knowledge sources (tables/graphs/metadata), and freshness-aware indexing. Enterprises are also expanding on-device and edge AI for lower latency, offline use, and reduced data exposure, while efficiency breakthroughs—distillation, quantization, speculative decodi

In [16]:
print(type(result.tasks_output))
print(len(result.tasks_output))

<class 'list'>
2


In [18]:
# Research Task Output
print(result.tasks_output[0])

- **Multimodal AI becomes mainstream**
  - Models increasingly handle **text + image + audio + video** in one system.
  - Stronger **vision-language** capabilities (reasoning over diagrams, charts, screenshots).
  - Growth in **speech-to-speech** and **video understanding** (summarization, event detection).
  - Practical trend: “AI copilots” that can interpret what users see/hear, not just what they type.

- **Agentic systems replace single-shot chat**
  - Shift from “answering questions” to **planning + tool use + iterative execution**.
  - Common pattern: LLM acts as a **controller** that calls external tools (search, code, databases, ticketing systems).
  - More **workflow automation** (research agents, support agents, sales ops agents).
  - Increasing focus on **reliability**: guardrails, state tracking, and self-checking loops.

- **Retrieval-Augmented Generation (RAG) evolves**
  - RAG remains a core architecture, but implementations are improving:
    - Better **chunking**, **hy

In [19]:
# Writer Task Output
print(result.tasks_output[1])

Across the industry, generative AI is rapidly evolving from simple text chat into practical, production-ready systems that combine multimodal understanding, agentic workflows, and stronger grounding. Multimodal models increasingly handle text, images, audio, and video together—enabling “AI copilots” that can interpret what users see or hear—not just what they type. In parallel, systems are shifting from single-shot responses to agentic architectures where an LLM plans, calls tools (search, code execution, databases, ticketing systems), tracks state, and iterates with reliability guardrails. Retrieval-Augmented Generation (RAG) remains central but is improving through better chunking, hybrid retrieval with reranking, structured knowledge sources (tables/graphs/metadata), and freshness-aware indexing. Enterprises are also expanding on-device and edge AI for lower latency, offline use, and reduced data exposure, while efficiency breakthroughs—distillation, quantization, speculative decodi

In [17]:
result

CrewOutput(raw='Across the industry, generative AI is rapidly evolving from simple text chat into practical, production-ready systems that combine multimodal understanding, agentic workflows, and stronger grounding. Multimodal models increasingly handle text, images, audio, and video together—enabling “AI copilots” that can interpret what users see or hear—not just what they type. In parallel, systems are shifting from single-shot responses to agentic architectures where an LLM plans, calls tools (search, code execution, databases, ticketing systems), tracks state, and iterates with reliability guardrails. Retrieval-Augmented Generation (RAG) remains central but is improving through better chunking, hybrid retrieval with reranking, structured knowledge sources (tables/graphs/metadata), and freshness-aware indexing. Enterprises are also expanding on-device and edge AI for lower latency, offline use, and reduced data exposure, while efficiency breakthroughs—distillation, quantization, sp